# P-value Manipulation

### Systematically alters mtcars data to change hypothesis test results from significant to non-significant, creating datasets for manipulation detection research.



### imports

In [6]:
import pandas as pd
import numpy as np
from scipy import stats

### Original p-value calculation

In [ ]:
df = pd.read_csv('My_data/mtcars.csv')

#  mpg ~ cyl
groups_orig = [df[df['cyl'] == g]['mpg'] for g in sorted(df['cyl'].unique())]
p1_orig = stats.f_oneway(*groups_orig).pvalue

# wt ~ am
group1_orig = df[df['am'] == df['am'].unique()[0]]['wt']
group2_orig = df[df['am'] == df['am'].unique()[1]]['wt']
p2_orig = stats.ttest_ind(group1_orig, group2_orig, equal_var=False).pvalue

# hp vs mpg 
p3_orig = stats.pearsonr(df['hp'], df['mpg'])[1]


### Helper function to manipulated the dataset

In [ ]:
def manipulation(df, col, group_col=None, test_type='anova', target_corr=None):
    df_mod = df.copy()
    changed = []
    indices = list(df_mod.index)
    for step in range(len(indices)):
        if test_type == 'anova':
            grand_mean = df_mod[col].mean()
            diffs = np.abs(df_mod[col] - grand_mean)
            idx_to_change = diffs.idxmax()
            group = df_mod.loc[idx_to_change, group_col]
            other_groups_mean = df_mod[df_mod[group_col] != group][col].mean()
            df_mod.at[idx_to_change, col] = other_groups_mean
            groups = [df_mod[df_mod[group_col] == g][col] for g in sorted(df_mod[group_col].unique())]
            p = stats.f_oneway(*groups).pvalue

        elif test_type == 'ttest':
            grand_mean = df_mod[col].mean()
            diffs = np.abs(df_mod[col] - grand_mean)
            idx_to_change = diffs.idxmax()
            group = df_mod.loc[idx_to_change, group_col]
            other_mean = df_mod[df_mod[group_col] != group][col].mean()
            df_mod.at[idx_to_change, col] = other_mean
            group1 = df_mod[df_mod[group_col] == df_mod[group_col].unique()[0]][col]
            group2 = df_mod[df_mod[group_col] == df_mod[group_col].unique()[1]][col]
            p = stats.ttest_ind(group1, group2, equal_var=False).pvalue

        elif test_type == 'corr':
            if target_corr is None:
                target_corr = df_mod['mpg'].mean()
            diffs = np.abs(df_mod[col] - target_corr)
            idx_to_change = diffs.idxmax()
            df_mod.at[idx_to_change, col] = target_corr
            p = stats.pearsonr(df_mod[col], df_mod['mpg'])[1]
        else:
            raise ValueError("test type not recognized.")
        
        changed.append({
            'idx': idx_to_change,
            'model': df_mod.loc[idx_to_change, 'model'],
            'orig_val': df.loc[idx_to_change, col],
            'new_val': df_mod.loc[idx_to_change, col],
            'pvalue': p
        })
        if p > 0.05:
            return changed, p
    return changed, p

In [ ]:
# 1. mpg ~ cyl 
manip1, p1 = manipulation(df, 'mpg', group_col='cyl', test_type='anova')
# 2. wt ~ am 
manip2, p2 = manipulation(df, 'wt', group_col='am', test_type='ttest')
# 3. hp vs mpg 
manip3, p3 = manipulation(df, 'hp', test_type='corr')

/var/folders/bw/twgt4mbj76n0dyvm8p9wyq900000gn/T/ipykernel_80997/3360788440.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '20.090625000000003' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_mod.at[idx_to_change, col] = target_corr


### Apply changes

In [10]:
df_combined = df.copy()

for m in manip1:
    df_combined.at[m['idx'], 'mpg'] = m['new_val']

for m in manip2:
    df_combined.at[m['idx'], 'wt'] = m['new_val']
 
for m in manip3:
    df_combined.at[m['idx'], 'hp'] = m['new_val']

/var/folders/bw/twgt4mbj76n0dyvm8p9wyq900000gn/T/ipykernel_80997/2006353968.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '20.090625000000003' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_combined.at[m['idx'], 'hp'] = m['new_val']


### Checking what was changed

In [11]:
print(f"Total mpg changes: {len(manip1)}")
print(f"Total wt changes: {len(manip2)}")
print(f"Total hp changes: {len(manip3)}")
print(f"Original p-values: mpg={p1_orig:.4f}, wt={p2_orig:.4f}, hp={p3_orig:.4f}")
print(f"Final p-values: mpg={p1:.4f}, wt={p2:.4f}, hp={p3:.4f}")

manipulation_mask = pd.DataFrame(False, index=df.index, columns=df.columns)

for m in manip1:
    manipulation_mask.at[m['idx'], 'mpg'] = True
for m in manip2:
    manipulation_mask.at[m['idx'], 'wt'] = True
for m in manip3:
    manipulation_mask.at[m['idx'], 'hp'] = True

print(f"\nTotal manipulated cells: {manipulation_mask.sum().sum()}")
print("\nManipulated cells per column:")
print(manipulation_mask.sum())

Total mpg changes: 17
Total wt changes: 8
Total hp changes: 5
Original p-values: mpg=0.0000, wt=0.0000, hp=0.0000
Final p-values: mpg=0.2177, wt=0.0685, hp=0.0864

Total manipulated cells: 30

Manipulated cells per column:
model     0
mpg      17
cyl       0
disp      0
hp        5
drat      0
wt        8
qsec      0
vs        0
am        0
gear      0
carb      0
dtype: int64
